## Notebook para atualização de tabelas no Postgres

Este notebook carrega as tabelas do projeto no Postgres de acordo com o schema de cada fase da arquitetura Medallion (RAW-TRUSTED-REFINED).

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
import json
import os
import numpy as np
from dotenv import load_dotenv # Requer: pip install python-dotenv

# Carrega variáveis de ambiente de um arquivo .env oculto
load_dotenv()

# Recupera credenciais
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_NAME = os.getenv("DB_NAME")
DB_PORT = os.getenv("DB_PORT", "5432")

if not all([DB_USER, DB_PASSWORD, DB_HOST, DB_NAME]):
    raise ValueError("Credenciais de banco não encontradas no arquivo .env")

# Conexão
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

try:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
        print("Conexão com o banco de dados estabelecida.")
except Exception as e:
    print(f"Falha na conexão: {e}")

Conexão com o banco de dados estabelecida.


## Verificar schemas

In [2]:
# Listar todos os schemas no banco de dados
with engine.connect() as conn:
    result = conn.execute(text(
        "SELECT schema_name FROM information_schema.schemata;"
    ))
    print([r[0] for r in result])

['public', 'information_schema', 'pg_catalog', 'refined', 'raw', 'trusted']


## Camada RAW

In [3]:
print("CARREGANDO TABELAS RAW NO POSTGRESQL RDS")

# ===== ARQUIVOS DE DADOS =====
PRODUCTS_RAW = '../data/raw/products_raw.json'
CUSTOMERS_RAW = '../data/raw/customers_raw.csv'
SALES_RAW = '../data/raw/sales_raw.csv'


# ===== VERIFICAR ARQUIVOS =====
print("\nVerificando arquivos...")
for f in [PRODUCTS_RAW, CUSTOMERS_RAW, SALES_RAW]:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024  # MB
        print(f"{f} ({size:.2f} MB)")
    else:
        print(f"{f} NÃO ENCONTRADO!")
        

# ===== CARREGAR ARQUIVOS =====
print("\nCarregando dados...")

try:
    with open(PRODUCTS_RAW, 'r') as f:
        products_df = pd.DataFrame(json.load(f))
        print(f"Products: {len(products_df):,} linhas")
        
        customers_df = pd.read_csv(CUSTOMERS_RAW)
        print(f"Customers: {len(customers_df):,} linhas")

        sales_df = pd.read_csv(SALES_RAW)
        print(f"Sales: {len(sales_df):,} linhas")
    
except Exception as e:
    print(f"Erro ao carregar: {e}")
    exit()

# ===== PREPARAÇÃO DO SCHEMA =====
print("\nPreparando Schema 'raw'...")

try:
    with engine.connect() as conn:
        # 1. GARANTIR QUE O SCHEMA EXISTE
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw;"))
        
        # 2. DROP COM PREFIXO 'raw.'
        conn.execute(text("DROP TABLE IF EXISTS raw.sales_raw CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS raw.products_raw CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS raw.customers_raw CASCADE;"))
        conn.commit()
    print("Schema verificado e tabelas limpas.")
except Exception as e:
    print(f"Erro ao preparar schema: {e}")
    exit()


# ===== INSERIR NO BANCO =====
print("\nInserindo no PostgreSQL (Schema: raw)...")

try:
    # 3. ADICIONADO O PARÂMETRO schema='raw'
    products_df.to_sql('products_raw', engine, if_exists='replace', index=False, schema='raw')
    print("products_raw inserida em 'raw'")
    
    customers_df.to_sql('customers_raw', engine, if_exists='replace', index=False, schema='raw')
    print("customers_raw inserida em 'raw'")
    
    sales_df.to_sql('sales_raw', engine, if_exists='replace', index=False, schema='raw')
    print("sales_raw inserida em 'raw'")
    
except Exception as e:
    print(f"Erro: {e}")
    exit()

# ===== CRIAR ÍNDICES =====
print("\nCriando índices...")

try:
    with engine.connect() as conn:
        # 4. ADICIONADO O PREFIXO 'raw.' NAS QUERIES SQL
        conn.execute(text("CREATE INDEX idx_products_id ON raw.products_raw(product_id);"))
        conn.execute(text("CREATE INDEX idx_customers_id ON raw.customers_raw(customer_id);"))
        conn.execute(text("CREATE INDEX idx_sales_id ON raw.sales_raw(sale_id);"))
        conn.execute(text("CREATE INDEX idx_sales_customer ON raw.sales_raw(customer_id);"))
        conn.execute(text("CREATE INDEX idx_sales_product ON raw.sales_raw(product_id);"))
        conn.commit()
    
    print("5 índices criados")
except Exception as e:
    print(f"Aviso nos índices: {e}")

# ===== VERIFICAR =====
print("\nDados no PostgreSQL (Schema raw):")

with engine.connect() as conn:
    # 5. SELECT TAMBÉM PRECISA DO 'raw.'
    r1 = conn.execute(text("SELECT COUNT(*) FROM raw.products_raw;")).scalar()
    r2 = conn.execute(text("SELECT COUNT(*) FROM raw.customers_raw;")).scalar()
    r3 = conn.execute(text("SELECT COUNT(*) FROM raw.sales_raw;")).scalar()

print(f"   • raw.products_raw: {r1:,} registros")
print(f"   • raw.customers_raw: {r2:,} registros")
print(f"   • raw.sales_raw: {r3:,} registros")

print("\n" + "="*80)
print("CARREGAMENTO CONCLUÍDO!")
print("="*80)

CARREGANDO TABELAS RAW NO POSTGRESQL RDS

Verificando arquivos...
../data/raw/products_raw.json (4.34 MB)
../data/raw/customers_raw.csv (0.70 MB)
../data/raw/sales_raw.csv (13.93 MB)

Carregando dados...
Products: 10,000 linhas
Customers: 5,000 linhas
Sales: 120,000 linhas

Preparando Schema 'raw'...
Schema verificado e tabelas limpas.

Inserindo no PostgreSQL (Schema: raw)...
products_raw inserida em 'raw'
customers_raw inserida em 'raw'
sales_raw inserida em 'raw'

Criando índices...
5 índices criados

Dados no PostgreSQL (Schema raw):
   • raw.products_raw: 10,000 registros
   • raw.customers_raw: 5,000 registros
   • raw.sales_raw: 120,000 registros

CARREGAMENTO CONCLUÍDO!


## Camada TRUSTED

In [4]:
print("CARGA DE DADOS 'TRUSTED' NO POSTGRESQL (LIMPANDO ARRAYS)")

# ===== ARQUIVOS DE DADOS =====
PRODUCTS_TRUSTED = '../data/trusted/products_trusted.parquet'
CUSTOMERS_TRUSTED = '../data/trusted/customers_trusted.parquet'
SALES_TRUSTED = '../data/trusted/sales_trusted.parquet'

# ============================================================================
# FUNÇÃO PARA LIMPAR COLUNAS COMPLEXAS (ARRAYS/LISTAS)
# ============================================================================
def limpar_dataframe_para_sql(df, nome_tabela):
    """
    Identifica e remove colunas que contêm listas, dicionários ou arrays numpy,
    pois elas quebram a inserção simples no SQL.
    """
    cols_to_drop = []
    
    print(f"\nAnalisando colunas de '{nome_tabela}'...")
    
    for col in df.columns:
        # Pega o primeiro valor não nulo para verificar o tipo
        sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
        
        if sample_val is not None:
            # Verifica se é lista, dicionário ou array numpy
            if isinstance(sample_val, (list, dict, np.ndarray, tuple)):
                cols_to_drop.append(col)
                
    if cols_to_drop:
        print(f"Removendo {len(cols_to_drop)} colunas complexas (Arrays/Listas):")
        for c in cols_to_drop:
            print(f"   - {c}")
        
        # Remove as colunas
        df_clean = df.drop(columns=cols_to_drop)
        return df_clean
    else:
        print("Nenhuma coluna complexa encontrada.")
        return df


# ===== PREPARAÇÃO DO SCHEMA =====
print("\nPreparando Schema 'trusted'...")

try:
    with engine.connect() as conn:
        # 1. GARANTIR QUE O SCHEMA EXISTE
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS trusted;"))
        
        # 2. DROP COM PREFIXO 'trusted.'
        conn.execute(text("DROP TABLE IF EXISTS trusted.sales_trusted CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS trusted.products_trusted CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS trusted.customers_trusted CASCADE;"))
        conn.commit()
    print("Schema verificado e tabelas limpas.")
except Exception as e:
    print(f"Erro ao preparar schema: {e}")
    exit()


# ===== CARREGAMENTO E LIMPEZA =====
dfs_to_upload = {}

try:
    # --- PRODUCTS ---
    if os.path.exists(PRODUCTS_TRUSTED):
        print(f"\nLendo {PRODUCTS_TRUSTED}...")
        df_prod = pd.read_parquet(PRODUCTS_TRUSTED)
        # Limpeza específica + automática
        df_prod = limpar_dataframe_para_sql(df_prod, "products_trusted")
        dfs_to_upload['products_trusted'] = df_prod
    else:
        print(f"Arquivo não encontrado: {PRODUCTS_TRUSTED}")

    # --- CUSTOMERS ---
    if os.path.exists(CUSTOMERS_TRUSTED):
        print(f"\nLendo {CUSTOMERS_TRUSTED}...")
        df_cust = pd.read_parquet(CUSTOMERS_TRUSTED)
        df_cust = limpar_dataframe_para_sql(df_cust, "customers_trusted")
        dfs_to_upload['customers_trusted'] = df_cust
    else:
        print(f"Arquivo não encontrado: {CUSTOMERS_TRUSTED}")

    # --- SALES ---
    if os.path.exists(SALES_TRUSTED):
        print(f"\nLendo {SALES_TRUSTED}...")
        df_sales = pd.read_parquet(SALES_TRUSTED)
        df_sales = limpar_dataframe_para_sql(df_sales, "sales_trusted")
        dfs_to_upload['sales_trusted'] = df_sales
    else:
        print(f"Arquivo não encontrado: {SALES_TRUSTED}")

except Exception as e:
    print(f"Erro crítico ao ler arquivos: {e}")
    exit()

# ===== UPLOAD PARA O BANCO =====
print("\n" + "="*40)
print("INICIANDO UPLOAD")
print("="*40)

try:
    products_df.to_sql('products_trusted', engine, if_exists='replace', index=False, schema='trusted')
    print("products_trusted inserida em 'trusted'")
    
    customers_df.to_sql('customers_trusted', engine, if_exists='replace', index=False, schema='trusted')
    print("customers_trusted inserida em 'trusted'")
    sales_df.to_sql('sales_trusted', engine, if_exists='replace', index=False, schema='trusted')
    print("sales_trusted inserida em 'trusted'")

except Exception as e:
    print(f"Erro: {e}")
    exit()

# ===== CRIAR ÍNDICES =====
print("\n Otimizando (Criando Índices)...")

try:
    with engine.connect() as conn:
        # Indices para Products
        if 'products_trusted' in dfs_to_upload:
            conn.execute(text("CREATE INDEX IF NOT EXISTS idx_prod_id ON products_trusted(product_id);"))
        
        # Indices para Customers
        if 'customers_trusted' in dfs_to_upload:
            conn.execute(text("CREATE INDEX IF NOT EXISTS idx_cust_id ON customers_trusted(customer_id);"))
            
        # Indices para Sales
        if 'sales_trusted' in dfs_to_upload:
            conn.execute(text("CREATE INDEX IF NOT EXISTS idx_sale_id ON sales_trusted(sale_id);"))
            conn.execute(text("CREATE INDEX IF NOT EXISTS idx_sale_cust ON sales_trusted(customer_id);"))
            conn.execute(text("CREATE INDEX IF NOT EXISTS idx_sale_prod ON sales_trusted(product_id);"))
            
        conn.commit()
    print("Índices criados.")
    
except Exception as e:
    print(f"Aviso na criação de índices: {e}")

# ===== VALIDAÇÃO =====
print("\nCONTAGEM FINAL:")
with engine.connect() as conn:
    for table in ['products_trusted', 'customers_trusted', 'sales_trusted']:
        try:
            count = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
            print(f"• {table}: {count:,} registros")
        except:
            print(f"• {table}: Não encontrada/Erro")

print(f"   • trusted.products_trusted: {r1:,} registros")
print(f"   • trusted.customers_trusted: {r2:,} registros")
print(f"   • trusted.sales_trusted: {r3:,} registros")

print("\n" + "="*80)
print("CARREGAMENTO CONCLUÍDO!")
print("="*80)



CARGA DE DADOS 'TRUSTED' NO POSTGRESQL (LIMPANDO ARRAYS)

Preparando Schema 'trusted'...
Schema verificado e tabelas limpas.

Lendo ../data/trusted/products_trusted.parquet...

Analisando colunas de 'products_trusted'...
Removendo 1 colunas complexas (Arrays/Listas):
   - technical_features

Lendo ../data/trusted/customers_trusted.parquet...

Analisando colunas de 'customers_trusted'...
Removendo 1 colunas complexas (Arrays/Listas):
   - expected_problems

Lendo ../data/trusted/sales_trusted.parquet...

Analisando colunas de 'sales_trusted'...
Nenhuma coluna complexa encontrada.

INICIANDO UPLOAD
products_trusted inserida em 'trusted'
customers_trusted inserida em 'trusted'
sales_trusted inserida em 'trusted'

 Otimizando (Criando Índices)...
Índices criados.

CONTAGEM FINAL:
• products_trusted: 10,000 registros
• customers_trusted: 5,000 registros
• sales_trusted: 120,000 registros
   • trusted.products_trusted: 10,000 registros
   • trusted.customers_trusted: 5,000 registros
   • tru

## Camada REFINED

In [5]:
print("CARGA DE DADOS 'REFINED' NO POSTGRESQL (LIMPANDO ARRAYS)")

# ===== ARQUIVOS DE DADOS =====
PRODUCTS_REFINED = '../data/refined/dim_products.parquet'
CUSTOMERS_REFINED = '../data/refined/dim_customers.parquet'
SALES_REFINED = '../data/refined/fact_sales.parquet'

# ============================================================================
# FUNÇÃO PARA LIMPAR COLUNAS COMPLEXAS (ARRAYS/LISTAS)
# ============================================================================
def limpar_dataframe_para_sql(df, nome_tabela):
    """
    Identifica e remove colunas que contêm listas, dicionários ou arrays numpy,
    pois elas quebram a inserção simples no SQL.
    """
    cols_to_drop = []
    
    print(f"\nAnalisando colunas de '{nome_tabela}'...")
    
    for col in df.columns:
        # Pega o primeiro valor não nulo para verificar o tipo
        sample_val = df[col].dropna().iloc[0] if not df[col].dropna().empty else None
        
        if sample_val is not None:
            # Verifica se é lista, dicionário ou array numpy
            if isinstance(sample_val, (list, dict, np.ndarray, tuple)):
                cols_to_drop.append(col)
                
    if cols_to_drop:
        print(f"Removendo {len(cols_to_drop)} colunas complexas (Arrays/Listas):")
        for c in cols_to_drop:
            print(f"   - {c}")
        
        # Remove as colunas
        df_clean = df.drop(columns=cols_to_drop)
        return df_clean
    else:
        print("Nenhuma coluna complexa encontrada.")
        return df


# ===== PREPARAÇÃO DO SCHEMA =====
print("\nPreparando Schema 'refined'...")

try:
    with engine.connect() as conn:
        # 1. GARANTIR QUE O SCHEMA EXISTE
        conn.execute(text("CREATE SCHEMA IF NOT EXISTS refined;"))
        
        # 2. DROP COM PREFIXO 'refined.'
        conn.execute(text("DROP TABLE IF EXISTS refined.fact_sales CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS refined.dim_products CASCADE;"))
        conn.execute(text("DROP TABLE IF EXISTS refined.dim_customers CASCADE;"))
        conn.commit()
    print("Schema verificado e tabelas limpas.")
except Exception as e:
    print(f"Erro ao preparar schema: {e}")
    exit()


# ===== CARREGAMENTO E LIMPEZA =====
dfs_to_upload = {}

try:
    # --- PRODUCTS ---
    if os.path.exists(PRODUCTS_REFINED):
        print(f"\nLendo {PRODUCTS_REFINED}...")
        df_prod = pd.read_parquet(PRODUCTS_REFINED)
        # Limpeza específica + automática
        df_prod = limpar_dataframe_para_sql(df_prod, "dim_products")
        dfs_to_upload['dim_products'] = df_prod
    else:
        print(f"Arquivo não encontrado: {PRODUCTS_REFINED}")

    # --- CUSTOMERS ---
    if os.path.exists(CUSTOMERS_REFINED):
        print(f"\nLendo {CUSTOMERS_REFINED}...")
        df_cust = pd.read_parquet(CUSTOMERS_REFINED)
        df_cust = limpar_dataframe_para_sql(df_cust, "dim_customers")
        dfs_to_upload['dim_customers'] = df_cust
    else:
        print(f"Arquivo não encontrado: {CUSTOMERS_REFINED}")

    # --- SALES ---
    if os.path.exists(SALES_REFINED):
        print(f"\nLendo {SALES_REFINED}...")
        df_sales = pd.read_parquet(SALES_REFINED)
        df_sales = limpar_dataframe_para_sql(df_sales, "fact_sales")
        dfs_to_upload['fact_sales'] = df_sales
    else:
        print(f"Arquivo não encontrado: {SALES_REFINED}")

except Exception as e:
    print(f"Erro crítico ao ler arquivos: {e}")
    exit()

# ===== UPLOAD PARA O BANCO =====
print("\n" + "="*40)
print("INICIANDO UPLOAD")
print("="*40)

try:
    dfs_to_upload['dim_products'].to_sql('dim_products', engine, if_exists='replace', index=False, schema='refined')
    print("dim_products inserida em 'refined'")
    
    dfs_to_upload['dim_customers'].to_sql('dim_customers', engine, if_exists='replace', index=False, schema='refined')
    print("dim_customers inserida em 'refined'")
    dfs_to_upload['fact_sales'].to_sql('fact_sales', engine, if_exists='replace', index=False, schema='refined')
    print("fact_sales inserida em 'refined'")

except Exception as e:
    print(f"Erro: {e}")
    exit()

# ===== CRIAR ÍNDICES =====
print("\nCriando índices...")

try:
    with engine.connect() as conn:
        # 4. ADICIONADO O PREFIXO 'refined.' NAS QUERIES SQL
        conn.execute(text("CREATE INDEX idx_products_id ON refined.dim_products(product_id);"))
        conn.execute(text("CREATE INDEX idx_customers_id ON refined.dim_customers(customer_id);"))
        conn.execute(text("CREATE INDEX idx_sales_id ON refined.fact_sales(sale_id);"))
        conn.execute(text("CREATE INDEX idx_sales_customer ON refined.fact_sales(customer_id);"))
        conn.execute(text("CREATE INDEX idx_sales_product ON refined.fact_sales(product_id);"))
        conn.commit()
    
    print("5 índices criados")
except Exception as e:
    print(f"Aviso nos índices: {e}")

# ===== VERIFICAR =====
print("\nDados no PostgreSQL (Schema refined):")

try:
    with engine.connect() as conn:
        # 5. SELECT TAMBÉM PRECISA DO 'refined.'
        r1 = conn.execute(text("SELECT COUNT(*) FROM refined.dim_products;")).scalar()
        r2 = conn.execute(text("SELECT COUNT(*) FROM refined.dim_customers;")).scalar()
        r3 = conn.execute(text("SELECT COUNT(*) FROM refined.fact_sales;")).scalar()

    print(f"• refined.dim_products: {r1:,} registros")
    print(f"• refined.dim_customers: {r2:,} registros")
    print(f"• refined.fact_sales: {r3:,} registros")
except Exception as e:
    print(f"Erro ao verificar registros: {e}")

print("\n" + "="*80)
print("CARREGAMENTO CONCLUÍDO!")
print("="*80)

CARGA DE DADOS 'REFINED' NO POSTGRESQL (LIMPANDO ARRAYS)

Preparando Schema 'refined'...
Schema verificado e tabelas limpas.

Lendo ../data/refined/dim_products.parquet...

Analisando colunas de 'dim_products'...
Removendo 2 colunas complexas (Arrays/Listas):
   - technical_features
   - supported_problems

Lendo ../data/refined/dim_customers.parquet...

Analisando colunas de 'dim_customers'...
Removendo 1 colunas complexas (Arrays/Listas):
   - expected_problems

Lendo ../data/refined/fact_sales.parquet...

Analisando colunas de 'fact_sales'...
Nenhuma coluna complexa encontrada.

INICIANDO UPLOAD
dim_products inserida em 'refined'
dim_customers inserida em 'refined'
fact_sales inserida em 'refined'

Criando índices...
5 índices criados

Dados no PostgreSQL (Schema refined):
• refined.dim_products: 10,000 registros
• refined.dim_customers: 5,000 registros
• refined.fact_sales: 120,000 registros

CARREGAMENTO CONCLUÍDO!


## Validação dos dados refined

In [6]:
import pandas as pd
from sqlalchemy import create_engine, text

print("VALIDANDO DADOS NO POSTGRESQL (AUDITORIA)")

DB_USER = "cryslayne"
DB_PASSWORD = "crys0231"
DB_HOST = "db-data-dadosfera.c9e0k2cgao73.sa-east-1.rds.amazonaws.com"
DB_PORT = "5432"
DB_NAME = "postgres"

# Lista de tabelas que você quer validar
TABELAS_ESPERADAS = ['dim_products', 'dim_customers', 'fact_sales']
SCHEMA = 'refined'

# ============================================================================
# EXECUÇÃO
# ============================================================================

try:
    # 1. Conexão
    db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(db_url)
    
    with engine.connect() as conn:
        print("Conectado ao Banco de Dados.\n")

        # Verifica cada tabela
        for tabela in TABELAS_ESPERADAS:
            print(f"ANALISANDO TABELA: '{SCHEMA}.{tabela}'")
            print("-" * 40)
            
            # CORREÇÃO 1: Usar information_schema.tables para verificar existência
            exists_query = text(f"""
                SELECT EXISTS (
                    SELECT FROM information_schema.tables 
                    WHERE table_schema = '{SCHEMA}' 
                    AND table_name = '{tabela}'
                );
            """)
            exists = conn.execute(exists_query).scalar()
            
            if not exists:
                print(f"ERRO: Tabela '{tabela}' NÃO ENCONTRADA no schema '{SCHEMA}'!")
                print("\n")
                continue

            # CORREÇÃO 2: Adicionar o prefixo do schema nas consultas de contagem e amostragem
            # 2. Volumetria (Count)
            count = conn.execute(text(f"SELECT COUNT(*) FROM {SCHEMA}.{tabela}")).scalar()
            print(f"Total de Registros: {count:,}")

            if count == 0:
                print("AVISO: A tabela existe mas está VAZIA.")
            else:
                # 3. Amostragem e Tipagem (Head)
                query = f"SELECT * FROM {SCHEMA}.{tabela} LIMIT 3"
                df_sample = pd.read_sql(query, conn)
                
                print("\nAmostra de Dados (Top 3):")
                # Nota: tabulate é necessário para o to_markdown()
                try:
                    print(df_sample.to_markdown(index=False))
                except ImportError:
                    print(df_sample) # Fallback caso tabulate não esteja instalado
                
                print("\nTb - Colunas detectadas:")
                print(f"   {list(df_sample.columns)}")

            print("\n" + "="*40 + "\n")

except Exception as e:
    print(f"Erro fatal na validação: {e}")

print("AUDITORIA CONCLUÍDA.")


VALIDANDO DADOS NO POSTGRESQL (AUDITORIA)
Conectado ao Banco de Dados.

ANALISANDO TABELA: 'refined.dim_products'
----------------------------------------
Total de Registros: 10,000

Amostra de Dados (Top 3):
| product_id   | product_name           | product_category      | product_subcategory   | manufacturer   | model   | bearing_type    | material   |   load_capacity |   max_speed |   temperature_limit | problem_type   |   unit_cost |   list_price | technical_description                                                                                         | lm_product_description                                                                                        |   problem_Contaminação |   problem_Desgaste |   problem_Superaquecimento |   problem_Vibração | full_description                                                                                                                                                                                             |
|:-------------|